In [1]:
import rioxarray as rx
import geopandas as gp
import shapely as shp
import pyproj
import numpy as np
import pycountry

import json

from emu_renewal.inputs import DATA_PATH

In [2]:
country = "Spain"
iso2_code = pycountry.countries.lookup(country).alpha_2
iso3_code = pycountry.countries.lookup(country).alpha_3

# Level is usually 1 for European countries and 2 for other, but check in facebook movement table
gadm_level = 1

In [3]:
# Gridded Population of the World 30sec 2020 dataset
# https://www.earthdata.nasa.gov/data/projects/gpw
pop_ds = rx.open_rasterio(DATA_PATH / "population/gpw_v4_population_count_rev11_2020_30_sec.tif")

# Alternative dataset from:
# https://github.com/lulingliu/GlobPOP
#pop_ds = rx.open_rasterio(DATA_PATH / "population/GlobPOP_Count_30arc_2020_I32.tiff")

In [ ]:
# Download the appropriate GADM boundaries json

import urllib.request

def get_source_url(iso3_code, gadm_level):
    return f"https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso3_code}_{gadm_level}.json.zip"

def get_local_file(iso3_code, gadm_level):
    return (DATA_PATH / f"population/gadm_input_json/gadm41_{iso3_code}_{gadm_level}.json.zip")

source = get_source_url(iso3_code, gadm_level)
dest = get_local_file(iso3_code, gadm_level)

urllib.request.urlretrieve(source, dest)

In [5]:
# GADM administrative boundaries as obtained from:
# https://gadm.org/download_country.html

# These files are directly downloadable as of 21/01/2025 via
# https://geodata.ucdavis.edu/gadm/gadm4.1/json/gadm41_{iso3_code}_{gadm_level}.json.zip

poly_df = gp.read_file(DATA_PATH / f"population/gadm_input_json/gadm41_{iso3_code}_{gadm_level}.json.zip")

In [6]:
# Set up the dimensions of a population cell
pix_dim = float((pop_ds.coords["x"][1] - pop_ds.coords["x"][0]).data)

In [7]:
def raster_to_polydf(raster_ds, data_name):
    pix_buf = (raster_ds.coords["x"][1] - raster_ds.coords["x"][0]).data * 0.5
    geoms = []
    mask_val = raster_ds.rio.nodata

    data = raster_ds.data[0]
    nvalid = (data != mask_val).sum()
    out_data = np.empty(nvalid)
    valid_idx = 0

    for ix, x in enumerate(raster_ds.coords["x"].data):
        for iy, y in enumerate(raster_ds.coords["y"].data):
            cell_data = data[iy,ix]
            if cell_data != mask_val:
                geoms.append(shp.Polygon([
                    (x-pix_buf, y-pix_buf),
                    (x+pix_buf, y-pix_buf),
                    (x+pix_buf, y+pix_buf),
                    (x-pix_buf, y+pix_buf),
                    ]
                    ))
                out_data[valid_idx] = cell_data
                valid_idx += 1
            
    data = data.flatten()
    
    df = gp.GeoDataFrame({data_name: out_data}, geometry=geoms)
    return df

In [8]:
# Stop GeoPandas complaining about our area intersection calculations
import warnings
warnings.filterwarnings("ignore", "Geometry is in a geographic CRS")

In [ ]:
pop_dict = {}

for ploc, poly_id in enumerate(poly_df[f"GID_{gadm_level}"]):
    print(poly_id)
    
    pop_bounds = np.array(poly_df.iloc[ploc].geometry.bounds)
    expanded_bounds = pop_bounds + np.array(([-pix_dim, -pix_dim, pix_dim, pix_dim]))
    dsc = pop_ds.rio.clip_box(*expanded_bounds)

    pop_df = raster_to_polydf(dsc, "population")

    # Ensure the GeoDataFrame has the correct projection
    pop_df = pop_df.set_crs(pyproj.CRS.from_wkt(pop_ds.rio.crs.to_wkt()))

    isect = gp.overlay(pop_df, poly_df.iloc[ploc:ploc+1], keep_geom_type=False)
    pop_dict[poly_id] = float(((isect.area / (pix_dim*pix_dim))*isect.population).sum())
    print(pop_dict[poly_id])
    

In [10]:
pop_dict = {k:float(v) for k,v in pop_dict.items()}

In [ ]:
sum(pop_dict.values())

In [12]:
json.dump(pop_dict, open(DATA_PATH / f"population/gadm_est/{iso3_code}_{gadm_level}.json","w"))